# Digital Signals Theory

In [1]:
import numpy as np

## Chapter 8: Fast Fourier transform

### Intuition into the algorithm

* While this chapter is very interesting, the best resource for intuition into the Fast Fourier transform algorithm is the [3. Divide & Conquer: FFT video](https://www.youtube.com/watch?v=iTMn0Kt18tg&t=16s&ab_channel=MITOpenCourseWare) in the lecture series for MIT 6.046J with Prof. Erik Demaine.
* The [Design And Analysis Of Algorithms site at OpenCourseWare@MIT](https://ocw.mit.edu/courses/6-046j-design-and-analysis-of-algorithms-spring-2015/) is also a trove of resources.

### Reference implementation of the radix-2 FFT
* See this answer to a question on Signal Processing @StackExchange: [implementing Prime-factor FFT algorithm](https://dsp.stackexchange.com/questions/78641/implementing-prime-factor-fft-algorithm/78645#78645)

### 8.3 Exercises

#### Exercise 8.1

_Optimize the radix-2 method to only compute the positive frequencies..._

First, let's have a look at the radix-2 implementation on pages 161-162.

In [2]:
def fft2(x):
    """
    Compute the DFT of an input x of N = 2**k samples.
    """
    N = len(x)

    if N == 1:
        return x
    else:
        X_even = fft2(x[0::2])
        X_odd  = fft2(x[1::2])

        X = np.zeros(N, dtype=np.complex128)

        for m in range(N):
            m_alias = m % (N//2)
            X[m] = X_even[m_alias] + np.exp(-2j * np.pi * m/N) * X_odd[m_alias]

    return X

##### idiot-check!

In [3]:
N = 16

x = np.random.rand(N) 

assert(np.allclose(fft2(x), np.fft.fft(x)))

<hr width=40%/>

Now let's hack that naive implmentation to come up with our own `fft2` that returns only the postive frequencies. This means those up to and including index $\frac{N}{2} + 1$.

In [4]:
def rfft2(x):
    """
    Compute the DFT of an input x of N = 2**k samples for only positive frequencies.
    """
    N = len(x)

    # in other words, we want to calculate frequncies up to N/2 + 1...
    end_idx = (N//2) + 1

    if N == 1:
        return x
    else:
        X_even = fft2(x[0::2])
        X_odd  = fft2(x[1::2])

        X = np.zeros(end_idx, dtype=np.complex128)

        for m in range(end_idx):
            if m < len(X_even):
                # no need for using aliasing in the final index of X
                idx = m
            else:
                # m_alias = m % (N//2), but in this case we know that m % (N//2) == 0
                idx = 0
            X[m] = X_even[idx] + np.exp(-2j * np.pi * m/N) * X_odd[idx]

    return X

##### idiot-check!

In [5]:
x = np.random.rand(N) 

assert(np.allclose(rfft2(x), np.fft.rfft(x)))

----

#### Exercise 8.2

_Given input $N = 3^{k}$ for some $k \in \mathbb{Z}$, modify the radix-2 method to obtain a radix-3 method._

In [6]:
def fft3(x):
    """
    Compute the DFT of an input x of N = 3**k samples.
    """
    N = len(x)

    if N == 1:
        return x
    else:
        X_0 = fft3(x[0::3])
        X_1 = fft3(x[1::3])
        X_2 = fft3(x[2::3])

        X = np.zeros(N, dtype=np.complex128)

        for m in range(N):
            m_alias = m % (N//3)
            X[m] = (
                X_0[m_alias] + 
                np.exp(-2j * np.pi * m/N) * X_1[m_alias] +  
                np.exp(-4j * np.pi * m/N) * X_2[m_alias]
            )

    return X

##### idiot-check!

In [7]:
N = 3**7

x = np.random.rand(N) 

assert(np.allclose(fft3(x), np.fft.fft(x)))